In [2]:
import json
import urllib.parse
import pandas as pd
from neuprint import Client, fetch_custom
from pathlib import Path

c:\Users\jsfdz\Documents\Python\cave_env\lib\site-packages\neuprint\utils.py:89: UserWarning: 
Progress bar will not work well in the notebook without ipywidgets.
Run the following commands (for notebook and jupyterlab users):

    conda install -c conda-forge ipywidgets
    jupyter nbextension enable --py widgetsnbextension
    jupyter labextension install @jupyter-widgets/jupyterlab-manager

...and then reload your jupyter session, and restart your kernel.

  warnings.warn(msg)


In [6]:
PROJECT_ROOT = Path.cwd()


IMPORT_DIR = PROJECT_ROOT / "output-MANCv1.2.3_04B_morphology_clusters_T1_20260407"

STATE_PATH = IMPORT_DIR / "MANC_v1.2.3_4B_morphology_clusters_reviewed20260407.json"

with open(STATE_PATH, "r") as f:
    state = json.load(f)

print("Loaded state:", state["title"])
print("Number of layers:", len(state["layers"]))


URL_PATH = IMPORT_DIR / "MANC_v1.2.3_4B_morphology_clusters_reviewed20260407_url.txt"

with open(URL_PATH, "r") as f:
    CLIO_URL = f.read().strip()

print("Loaded URL:")
print(CLIO_URL)


#create the subfolder to store tables as subfolder of the import folder
TABLES_DIR = IMPORT_DIR / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)


print("Tables output:", TABLES_DIR.resolve())
print("Import from:", IMPORT_DIR.resolve())

Loaded state: MANC_v1.2.3_4B_morphology_clusters_JHS
Number of layers: 21
Loaded URL:
https://clio-ng.janelia.org/#!%7B%22title%22%3A%22MANC_v1.2.3_4B_morphology_clusters_JHS%22%2C%22dimensions%22%3A%7B%22x%22%3A%5B8e-09%2C%22m%22%5D%2C%22y%22%3A%5B8e-09%2C%22m%22%5D%2C%22z%22%3A%5B8e-09%2C%22m%22%5D%7D%2C%22position%22%3A%5B23056.5%2C29733.5%2C41138.5%5D%2C%22crossSectionOrientation%22%3A%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22crossSectionScale%22%3A1%2C%22projectionOrientation%22%3A%5B0%2C0.7071067690849304%2C-0.7071067690849304%2C0%5D%2C%22projectionScale%22%3A91364.04452716278%2C%22layout%22%3A%223d%22%2C%22showSlices%22%3Afalse%2C%22projectionBackgroundColor%22%3A%22%23ffffff%22%2C%22crossSectionBackgroundColor%22%3A%22%23ffffff%22%2C%22selectedLayer%22%3A%7B%22flex%22%3A1.49%2C%22size%22%3A426%2C%22visible%22%3Atrue%2C%22layer%22%3A%22manc%3Av1.2.3%22%7D%2C%22settingsPanel%22%3A%7B%22visible%22%3Atrue%7D%2C%22layers%22%3A%5B%7B%22type%22%3A%22image%22%2C%22nam

In [7]:
#decode the state from the link
#CLIO_URL is read in above
encoded_state = CLIO_URL.split("#!", 1)[1]
state = json.loads(urllib.parse.unquote(encoded_state))

print(state.keys())
print("Number of layers:", len(state.get("layers", [])))

dict_keys(['title', 'dimensions', 'position', 'crossSectionOrientation', 'crossSectionScale', 'projectionOrientation', 'projectionScale', 'layout', 'showSlices', 'projectionBackgroundColor', 'crossSectionBackgroundColor', 'selectedLayer', 'settingsPanel', 'layers'])
Number of layers: 21


In [8]:
#inspect layer names
for i, layer in enumerate(state.get("layers", [])):
    print(i, layer.get("type"), layer.get("name"))

0 image EM
1 segmentation MANC_v1.2.3_BA
2 segmentation MANC_v1.2.3_BI
3 segmentation MANC_v1.2.3_BR
4 segmentation MANC_v1.2.3_IA
5 segmentation MANC_v1.2.3_IR
6 segmentation MANC_v1.2.3_br1
7 segmentation MANC_v1.2.3_br2
8 segmentation MANC_v1.2.3_br3
9 segmentation MANC_v1.2.3_ir_2_04B_1
10 segmentation MANC_v1.2.3_ir_2_04B_2
11 segmentation MANC_v1.2.3_ir_2_04B_3
12 segmentation MANC_v1.2.3_ir_2_04B_4
13 segmentation MANC_v1.2.3_ir_2_04B_5
14 segmentation MANC_v1.2.3_ir_2_04B_6
15 segmentation MANC_v1.2.3_ir_1_1
16 segmentation MANC_v1.2.3_ir_1_2
17 segmentation MANC_v1.2.3_ir_1_3
18 segmentation MANC_v1.2.3_ir_1_4
19 segmentation MANC_v1.2.3_ir_1_5
20 segmentation MANC_v1.2.3_ir_1_6


In [9]:
#Option A: all segmentation layers
rows = []

for layer in state.get("layers", []):
    if layer.get("type") != "segmentation":
        continue

    layer_name = layer.get("name", "")
    segments = layer.get("segments", []) or []

    for seg_id in segments:
        rows.append({
            "layer_name": layer_name,
            "segment_id": int(seg_id)
        })

segments_df = (
    pd.DataFrame(rows)
    .drop_duplicates()
    .sort_values(["layer_name", "segment_id"])
    .reset_index(drop=True)
)

segments_df.head()

,layer_name,segment_id
0,MANC_v1.2.3_BA,17490
1,MANC_v1.2.3_BI,11143
2,MANC_v1.2.3_BI,14031
3,MANC_v1.2.3_BI,15393
4,MANC_v1.2.3_BI,15401


In [27]:
# # only select layers, select by name 
# TARGET_PREFIX = "MANC_v1.2.3_"

# rows = []

# for layer in state.get("layers", []):
#     if layer.get("type") != "segmentation":
#         continue

#     layer_name = layer.get("name", "")
#     if not layer_name.startswith(TARGET_PREFIX):
#         continue

#     for seg_id in layer.get("segments", []) or []:
#         rows.append({
#             "layer_name": layer_name,
#             "segment_id": int(seg_id)
#         })

# segments_df = (
#     pd.DataFrame(rows)
#     .drop_duplicates()
#     .sort_values(["layer_name", "segment_id"])
#     .reset_index(drop=True)
# )

# segments_df.head()

In [10]:
#connect to neuPrint
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from current working directory

token = os.environ["NEUPRINT_APPLICATION_CREDENTIALS"]

from neuprint import Client

c = Client(
    "https://neuprint.janelia.org",
    dataset="manc:v1.2.3",
    token=token
)
print(c)

Client("https://neuprint.janelia.org", "manc:v1.2.3")


In [11]:
#fetch data for only the IDs inteh state
body_ids = sorted(segments_df["segment_id"].unique().tolist())
len(body_ids)

cypher = f"""
MATCH (n:Neuron)
WHERE n.bodyId IN {body_ids}
RETURN
    n.bodyId AS bodyId,
    n.type AS type,
    n.instance AS instance,
    n.class AS class,
    n.subclass AS subclass,
    n.description AS description,
    n.serialMotif AS serialMotif
ORDER BY bodyId
"""

neurons_df = fetch_custom(cypher, client=c)
neurons_df.head()

,bodyId,type,instance,class,subclass,description,serialMotif
0,10024,IN04B041,IN04B041_T1_L,intrinsic neuron,BR,4B in T1 LHS,None
1,10314,AN04B001,AN04B001_T1_L,ascending neuron,IA,4B? primary in T1 LHS,complex
2,11143,IN04B002,IN04B002_T1_L,intrinsic neuron,BI,pred ChAT,None
3,12523,IN04B008,IN04B008_T1_L,intrinsic neuron,BR,4B? in T1 LHS,independent leg
4,13128,IN04B025,IN04B025_T1_L,intrinsic neuron,IR,20A in T1 LHS,independent leg


In [12]:
#merge layer IDs with MANC annotations
final_table = (
    segments_df
    .merge(
        neurons_df,
        left_on="segment_id",
        right_on="bodyId",
        how="left"
    )
    .sort_values(["layer_name", "segment_id"])
    .reset_index(drop=True)
)

final_table.head()

,layer_name,segment_id,bodyId,type,instance,class,subclass,description,serialMotif
0,MANC_v1.2.3_BA,17490,17490,AN04B051,AN04B051_T1_L,ascending neuron,BA,pred ChAT,None
1,MANC_v1.2.3_BI,11143,11143,IN04B002,IN04B002_T1_L,intrinsic neuron,BI,pred ChAT,None
2,MANC_v1.2.3_BI,14031,14031,IN04B028,IN04B028_T1_L,intrinsic neuron,BI,None,None
3,MANC_v1.2.3_BI,15393,15393,IN04B024,IN04B024_T1_L,intrinsic neuron,BI,20A? in T1 LHS,None
4,MANC_v1.2.3_BI,15401,15401,IN04B010,IN04B010_T1_L,intrinsic neuron,BI,20A in T1 LHS,None


In [13]:
#select columns of interest
final_table = final_table[[
    "layer_name",
    "segment_id",
    "type",
    "instance",
    "class",
    "subclass",
    "description",
    "serialMotif"
]]

final_table

,layer_name,segment_id,type,instance,class,subclass,description,serialMotif
0,MANC_v1.2.3_BA,17490,AN04B051,AN04B051_T1_L,ascending neuron,BA,pred ChAT,None
1,MANC_v1.2.3_BI,11143,IN04B002,IN04B002_T1_L,intrinsic neuron,BI,pred ChAT,None
2,MANC_v1.2.3_BI,14031,IN04B028,IN04B028_T1_L,intrinsic neuron,BI,None,None
3,MANC_v1.2.3_BI,15393,IN04B024,IN04B024_T1_L,intrinsic neuron,BI,20A? in T1 LHS,None
4,MANC_v1.2.3_BI,15401,IN04B010,IN04B010_T1_L,intrinsic neuron,BI,20A in T1 LHS,None
...,...,...,...,...,...,...,...,...
178,MANC_v1.2.3_ir_2_04B_4,102138,IN04B085,IN04B085_T1_L,intrinsic neuron,IR,4B? in T1 LHS,None
179,MANC_v1.2.3_ir_2_04B_4,158391,IN04B100,IN04B100_T1_L,intrinsic neuron,IR,4B in T1 LHS,independent leg
180,MANC_v1.2.3_ir_2_04B_5,18641,IN04B069,IN04B069_T1_L,intrinsic neuron,IR,None,None
181,MANC_v1.2.3_ir_2_04B_5,20836,IN04B078,IN04B078_T1_L,intrinsic neuron,IR,4B in T1 LHS,independent leg


In [14]:
#summary counts per layer
final_table.groupby("layer_name")["segment_id"].nunique().sort_values(ascending=False)

layer_name
MANC_v1.2.3_IR            60
MANC_v1.2.3_BR            26
MANC_v1.2.3_ir_2_04B_2    14
MANC_v1.2.3_br1           13
MANC_v1.2.3_br2            9
MANC_v1.2.3_ir_1_3         7
MANC_v1.2.3_ir_1_4         7
MANC_v1.2.3_BI             6
MANC_v1.2.3_ir_2_04B_4     6
MANC_v1.2.3_ir_1_1         6
MANC_v1.2.3_ir_2_04B_1     5
MANC_v1.2.3_IA             4
MANC_v1.2.3_ir_1_2         4
MANC_v1.2.3_br3            4
MANC_v1.2.3_ir_1_6         4
MANC_v1.2.3_ir_1_5         2
MANC_v1.2.3_ir_2_04B_5     2
MANC_v1.2.3_ir_2_04B_3     2
MANC_v1.2.3_BA             1
MANC_v1.2.3_ir_2_04B_6     1
Name: segment_id, dtype: int64

In [17]:
#select layers
selected_layer_names = [
    "MANC_v1.2.3_BA",
    "MANC_v1.2.3_IA",
    "MANC_v1.2.3_BI",

    "MANC_v1.2.3_br1",
    "MANC_v1.2.3_br2",
    "MANC_v1.2.3_br3",

    "MANC_v1.2.3_ir_2_04B_1",
    "MANC_v1.2.3_ir_2_04B_2",
    "MANC_v1.2.3_ir_2_04B_3",
    "MANC_v1.2.3_ir_2_04B_4",
    "MANC_v1.2.3_ir_2_04B_5",
    "MANC_v1.2.3_ir_2_04B_6",

    "MANC_v1.2.3_ir_1_1",
    "MANC_v1.2.3_ir_1_2",
    "MANC_v1.2.3_ir_1_3",
    "MANC_v1.2.3_ir_1_4",
    "MANC_v1.2.3_ir_1_5",
    "MANC_v1.2.3_ir_1_6",
    

]

selected_final_table = (
    final_table[final_table["layer_name"].isin(selected_layer_names)]
    .copy()
    .sort_values(["layer_name", "segment_id"])
    .reset_index(drop=True)
)

selected_final_table


,layer_name,segment_id,type,instance,class,subclass,description,serialMotif
0,MANC_v1.2.3_BA,17490,AN04B051,AN04B051_T1_L,ascending neuron,BA,pred ChAT,None
1,MANC_v1.2.3_BI,11143,IN04B002,IN04B002_T1_L,intrinsic neuron,BI,pred ChAT,None
2,MANC_v1.2.3_BI,14031,IN04B028,IN04B028_T1_L,intrinsic neuron,BI,None,None
3,MANC_v1.2.3_BI,15393,IN04B024,IN04B024_T1_L,intrinsic neuron,BI,20A? in T1 LHS,None
4,MANC_v1.2.3_BI,15401,IN04B010,IN04B010_T1_L,intrinsic neuron,BI,20A in T1 LHS,None
...,...,...,...,...,...,...,...,...
92,MANC_v1.2.3_ir_2_04B_4,102138,IN04B085,IN04B085_T1_L,intrinsic neuron,IR,4B? in T1 LHS,None
93,MANC_v1.2.3_ir_2_04B_4,158391,IN04B100,IN04B100_T1_L,intrinsic neuron,IR,4B in T1 LHS,independent leg
94,MANC_v1.2.3_ir_2_04B_5,18641,IN04B069,IN04B069_T1_L,intrinsic neuron,IR,None,None
95,MANC_v1.2.3_ir_2_04B_5,20836,IN04B078,IN04B078_T1_L,intrinsic neuron,IR,4B in T1 LHS,independent leg


In [19]:
layer_notes = {
    "MANC_v1.2.3_ir_2_04B_1": "IR 04B morphology group 1",
    "MANC_v1.2.3_ir_2_04B_2": "IR 04B morphology group 2",
    "MANC_v1.2.3_ir_2_04B_3": "IR 04B anterior neuropil branch group",
    "MANC_v1.2.3_ir_2_04B_4": "IR 04B caudal neuropil branch group",
    "MANC_v1.2.3_ir_2_04B_5": "IR 04B small distinct morphology group",
    "MANC_v1.2.3_ir_2_04B_6": "IR 04B singleton",
    "MANC_v1.2.3_ir_1_1": "IR MANC tag 20A independent leg group",
    "MANC_v1.2.3_ir_1_2": "IR MANC tag 20A20A morphology group 2",
    "MANC_v1.2.3_ir_1_3": "IR MANC tag 20A20A morphology group 3",
    "MANC_v1.2.3_ir_1_4": "IR MANC tag 20A20A morphology group 4",
    "MANC_v1.2.3_ir_1_5": "IR MANC tag 20A20A small group",
    "MANC_v1.2.3_ir_1_6": "IR MANC tag 20A20A morphology group 6",
    "MANC_v1.2.3_br1": "BR morphology group 1",
    "MANC_v1.2.3_br2": "BR MANC TAG 20A morphology group 2",
    "MANC_v1.2.3_br3": "BR morphology group 3",
    "MANC_v1.2.3_BA": "BA morphology",
    "MANC_v1.2.3_BI": "BI morphology",
    "MANC_v1.2.3_IA": "IA morphology",
}

selected_final_table["description JHS"] = selected_final_table["layer_name"].map(layer_notes)

anatomical_layer_notes = {
    "MANC_v1.2.3_ir_2_04B_1": "caudal  medial nuclei, confined  crecent-shape crandio-caudal of medial side",
    "MANC_v1.2.3_ir_2_04B_2": "dispersed soma, fasiculation weight lateral side from dorsal to ventral",
    "MANC_v1.2.3_ir_2_04B_3": "caudal soma, strong primary neurite",
    "MANC_v1.2.3_ir_2_04B_4": "cranial soma, three strong primary neurites",
    "MANC_v1.2.3_ir_2_04B_5": "caudal soma, medial fasicualtion, extension to lateral, there too fasiculation",
    "MANC_v1.2.3_ir_2_04B_6": "IR 04B singleton",
    "MANC_v1.2.3_ir_1_1": "dispersed soma position, long neurite, fuzzy fasiculation covers neuropil",
    "MANC_v1.2.3_ir_1_2": "dispersed soma, long neurite, fasiculation covers neuroIR MANC tag 20A20A morphology group 2",
    "MANC_v1.2.3_ir_1_3": "cranial soma, thick neurite separating cranial and caudal fasiculation 3",
    "MANC_v1.2.3_ir_1_4": "cranial soma, long neurite, lateral fasiculation",
    "MANC_v1.2.3_ir_1_5": "Very similar to  IR_1_4, fasiculation weight is more to the neuropill core",
    "MANC_v1.2.3_ir_1_6": "Very silolar to IR 1_4 and IR_1_5 but with extra medial branch that does not fasiculate strongly",
    "MANC_v1.2.3_br1": "caudal medial nuclei, faciculation  crecent-shape crandio-caudal of medial side",
    "MANC_v1.2.3_br2": "lateral nuclei, strong primary neurines; form chiasm",
    "MANC_v1.2.3_br3": "medial nuclei, fuzzy neurites all over neuropil and towards contralateral side",
    "MANC_v1.2.3_BA": "stretchy thin neuron, extends anterior and posterior outside of T1",
    "MANC_v1.2.3_BI": "fuzzy and extends into T2 and T3",
    "MANC_v1.2.3_IA": "ascending and desendign neurites- outside of neuropil",
}

selected_final_table["anatomy"] = selected_final_table["layer_name"].map(anatomical_layer_notes)

In [20]:
#save 
OUT_FILE = TABLES_DIR / "04BT1_clio_state_segment_table.csv"

selected_final_table.to_csv(OUT_FILE, index=False)

print("Saved to:", OUT_FILE.resolve())

Saved to: C:\Users\jsfdz\4B-analysis\4B-analysis\output-MANCv1.2.3_04B_morphology_clusters_T1_20260407\Tables\04BT1_clio_state_segment_table.csv
